# Annotations are not an input firewall

QuickCart is our fictional online shop. Its made-up support tickets carry an ID, an amount and some tags, and each one has to follow clear rules.

> **Your goal:** Tell type hints apart from runtime validation and from rules about a whole batch.<br>
> **You need:** Basic functions and collections; the validation warm-up first if your route included it.<br>
> **Effort:** 30–45 minutes (estimate).<br>
> **You will produce:** Validated tickets, a test that catches `True` used as an amount, and checks for duplicate IDs.

## Before executing

Work in your own copy, in a notebook editor running Python 3.11. Running it in Colab has not been tested, and hosted notebooks may use a different Python version. You need no extra packages, keys, paid model or real customer data. Keep your work and notes private. Nothing here starts a server or uses a paid API; stop your kernel when you finish.

Start from the cells as supplied, not from an old kernel. Predict, run the cells in order, add the fault described, restore the rule and rerun the same check. Restart before your final in-order run. The notebook starts with all outputs cleared. Note that an unfinished TODO function can sit there while "Run all" still passes; you must call your own function separately to test it.

[Previous: validation warm-up](01-validation-concepts.ipynb) · [Bridge route](README.md) · [Concept explanation](../../study-guides/python-engineering-bridge/README.md)

## Predict: who enforces the contract?

QuickCart receives an amount as text, or as `True`. A type hint labels the contract but does not check it while the code runs. **Predict:** will `hint_only("ten")` be rejected? If so, what do you expect to do the rejecting?

Run the next cell once you have predicted. Its assert passes, because the function simply returns the string unchanged. This is deliberate misuse to show how Python behaves at runtime; it is not how you should write application code. A static type checker could flag it, but this cell does not run one.

In [ ]:
def hint_only(value: int) -> int:
    return value
assert hint_only("ten") == "ten"  # Intentional type misuse: runtime accepts it.


## Build the domain boundary

A **dataclass** writes routine methods for you, such as the initializer, from the fields you declare. `__post_init__` runs straight after that initializer, and this example uses it to check the values. `frozen=True` stops anyone reassigning the attributes, but it does not freeze mutable objects stored inside. Storing tags as a tuple of strings keeps this model simple.

Read and run the model cell, then find the check that enforces each rule:

| Field | Rule |
|---|---|
| `ticket_id` | Text that is not blank, at most 64 characters |
| `amount` | A whole number from 0 to 1,000,000 teaching units, never `True` or `False` |
| `tags` | A tuple of at most 10 non-blank strings, each at most 40 characters |

`InputError` is this project's own kind of `ValueError`, so a rejected ticket can be told apart from an unexpected bug. Creating a `Ticket` directly must follow the same rules as parsing input. A valid ticket still does not authorise a refund.

In [ ]:
from dataclasses import dataclass


class InputError(ValueError):
    """A declared synthetic ticket contract failed; messages never echo raw payloads."""


@dataclass(frozen=True)
class Ticket:
    ticket_id: str
    amount: int
    tags: tuple[str, ...] = ()

    def __post_init__(self) -> None:
        if not isinstance(self.ticket_id, str) or not self.ticket_id.strip() or len(self.ticket_id) > 64:
            raise InputError('Ticket ID must be nonempty text of at most 64 characters')
        if type(self.amount) is not int or not 0 <= self.amount <= 1_000_000:
            raise InputError('Amount must be an integer in synthetic units, 0–1000000')
        if not isinstance(self.tags, tuple) or len(self.tags) > 10:
            raise InputError('Tags must be a tuple with at most ten values')
        if any(not isinstance(tag, str) or not tag.strip() or len(tag) > 40 for tag in self.tags):
            raise InputError('Tags must be nonempty strings of at most 40 characters')




## Parse external data without silently changing its meaning

The parser first checks that the input is a dictionary with the required fields. The set difference `payload.keys() - {'ticket_id', 'amount', 'tags'}` finds any unknown fields. It then checks that the incoming tags are a list before turning them into a tuple and creating the `Ticket`.

**Predict:** why reject `tags="shipping"` instead of just calling `tuple(tags)`? Because converting a string gives you its individual characters, which hides the fact that the input had the wrong shape. The parser checks the format of incoming data; the model checks its own rules even when a ticket is created directly.

In the worked example below, the amount stays **10 units** and the list `["shipping"]` becomes `("shipping",)`. If tags are missing, they default to an empty list before conversion. Run the checks, then add one valid case with an amount of zero and one case that is rejected for an unknown field.

In [ ]:
def parse_ticket(payload: object) -> Ticket:
    if not isinstance(payload, dict) or not {'ticket_id', 'amount'} <= payload.keys():
        raise InputError('Ticket object requires ticket_id and amount')
    if payload.keys() - {'ticket_id', 'amount', 'tags'}:
        raise InputError('Unknown ticket fields')
    tags = payload.get('tags', [])
    if not isinstance(tags, list):
        raise InputError('External tags must be a list')
    return Ticket(payload['ticket_id'], payload['amount'], tuple(tags))




In [ ]:
ticket = parse_ticket({"ticket_id": "QC-1", "amount": 10, "tags": ["shipping"]})
assert ticket.amount == 10 and ticket.tags == ("shipping",)
for payload in ({"ticket_id":"QC-1","amount":True}, {"ticket_id":"QC-1","amount":"10"}, {}):
    try:
        parse_ticket(payload)
    except InputError:
        pass
    else:
        raise AssertionError("Invalid input accepted")


## Break, diagnose and repair

In your own copy, replace the model's amount check:

```python
if type(self.amount) is not int or not 0 <= self.amount <= 1_000_000:
```

with:

```python
if not 0 <= self.amount <= 1_000_000:
```

Predict what will happen, then rerun the **model cell, the parser cell and the test cell, in that order**. The `True` case should now raise `AssertionError: Invalid input accepted`, because `True` counts as 1, which is inside the allowed range. Put the exact-type check back and rerun the same cells. Defining the dataclass again matters: an old `ticket` object still belongs to the old class, so create your test objects again too.

**Decision:** here, text amounts are rejected. If an upstream system can only send text, write and test a separate parsing rule for it; do not expect type hints to make that decision for you.

## Independent practice: validate the whole batch

Write `parse_batch` below. It takes a list of payloads, parses each one with `parse_ticket`, keeps them in their original order and raises `InputError` if a ticket ID appears twice. Duplicate tags inside one ticket are still allowed. Compare IDs exactly as stored: this exercise does not trim spaces or ignore upper and lower case.

<details>
<summary>After your attempt: open the hints one by one</summary>

1. Keep the output list, plus a separate set of the IDs you have already seen.
2. Parse each payload before you accept its ID.
3. If an ID has been seen before, raise an error; never quietly drop the duplicate.
4. Add accepted tickets in input order. If a later item is invalid, raise an error rather than returning half a result.

**Why:** two tickets that are each valid can still clash when you look at the batch as a whole. Checking shape, checking each item, checking the whole batch and checking permissions are four different jobs.

</details>

Once your function is written, add a code cell below it and run:

```python
payloads = [
    {"ticket_id": "QC-B", "amount": 10, "tags": ["shipping", "shipping"]},
    {"ticket_id": "QC-A", "amount": 20},
]
parsed = parse_batch(payloads)
assert [ticket.ticket_id for ticket in parsed] == ["QC-B", "QC-A"]
assert parsed[0].tags == ("shipping", "shipping")
assert parse_batch([]) == []

for invalid in [
    [payloads[0], payloads[0]],
    [payloads[0], {"ticket_id": "QC-C", "amount": True}],
    [{"ticket_id": "QC-C"}],
]:
    try:
        parse_batch(invalid)
    except InputError:
        pass
    else:
        raise AssertionError("Conflicting or malformed batch accepted")
```

Until you write the function, you will see `NotImplementedError`; that is not a successful rejection. Add one more invalid-field case of your own, and explain why the reference checks alone cannot prove your function is correct.

In [ ]:
def parse_batch(payloads: list[object]) -> list[Ticket]:
    # TODO: parse each payload with parse_ticket and keep input order.
    # TODO: raise InputError for a repeated ticket_id; duplicate tags inside one ticket stay allowed.
    raise NotImplementedError("Independent extension")

# After implementing, add normal, duplicate-ID, duplicate-tag and empty-batch assertions here.
# Do not call this cell complete because Run all passes: the stub is defined, not exercised.


## Troubleshoot and transfer

| Symptom | Likely cause | What to check and do |
|---|---|---|
| `NameError` | A cell that defines something did not run | Restart and run the earlier cells in order. |
| Your edited model seems unchanged | The old class or an old object is still in memory | Rerun the model and parser cells and create your test tickets again. |
| The duplicate test passes when it should fail | Duplicates were quietly dropped, or the unfinished function was never called | Check that the test cell calls your function and expects `InputError`. |
| `NotImplementedError` | Your function is still the unfinished stub | Write it, then rerun the checks. |

Save your first prediction, the failure, your fix, your own tests and any help you used. Clear sensitive output before sharing and stop your kernel when you finish; nothing here starts a server or costs money.

**Reflect:** which check belongs in the parser, which in the ticket and which in the batch? After your selected notebooks, you will rebuild these boundaries in the required [package project](python-bridge/README.md). Reading the reference solution is useful help, but say so when you assess your own work.

## Check understanding and finish

**Pause and decide:** two tickets are each valid on their own but have the same ID. Which boundary should catch the clash?

<details>
<summary>Compare your reasoning</summary>

The batch check must catch duplicate IDs. A single ticket cannot compare itself with the rest of the batch, and quietly dropping one record changes the rules.

</details>

Before you call this done, keep:

- Your prediction and the real reference result from a fresh, in-order run.
- The deliberately failing check, your fix and the check passing again.
- Asserts that really call your own function, including invalid input.
- A short explanation of the boundary, and any hints or AI help you used.

**You can now:** tell type hints apart from runtime validation and from batch-level rules, and prove it with checks rather than a saved green result.

**Next:** open [the next unit](03-async-and-exceptions.ipynb) if your route includes it; otherwise go to your next selected unit in the [bridge map](README.md#unit-map-and-estimated-effort). The [package checkpoint](python-bridge/README.md) is still required before the API project. Save your evidence, clear sensitive output before sharing and stop your session.